<a href="https://colab.research.google.com/github/Danyal-Nadeem/flyrank-ml-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Danyal-Nadeem/flyrank-ml-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
One row = one page's monthly search performance snapshot, joined to its
client via client_hash_id. Time window: I'm working with month=2026-03
as my analysis month (per the assignment's guidance to avoid the sealed
final-month sample table).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
Feature: impressions, clicks, avg_position, ctr (observable before the
decision point)
Label: is_declining (derived from trend direction — the outcome I'm
predicting)
Context: client_hash_id, access_profile, has_gsc_access (needed to filter/
join but not fed to the model)
Excluded: raw query-level keyword strings — too granular for page-level
scoring, and re-identification risk per the data-use terms.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Danyal-Nadeem/flyrank-ml-starter"
REPO_DIR = "flyrank-ml-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

from google.colab import userdata
from huggingface_hub import login, HfApi
import duckdb

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token

# Discover actual files in the dataset
api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
print("=== Files in warehouse ===")
for f in files:
    print(f)

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"SET s3_endpoint='huggingface.co';")

# Grain check: row count and date span on a mid-panel month
try:
    grain = con.sql("""
        SELECT COUNT(*) AS row_count
        FROM 'hf://datasets/FlyRank/internship-warehouse/**/*.parquet'
        WHERE month = '2026-03'
    """).df()
    print("\n=== Row count for month=2026-03 ===")
    print(grain)
except Exception as e:
    print("Adjust table path once file list above is known:", e)

# Availability check
try:
    avail = con.sql("""
        SELECT COUNT(*) AS available_rows
        FROM 'hf://datasets/FlyRank/internship-warehouse/**/*.parquet'
        WHERE month = '2026-03' AND is_active IS TRUE
    """).df()
    print("\n=== Available (is_active) rows ===")
    print(avail)
except Exception as e:
    print("Adjust once schema confirmed:", e)

# Missing values check
try:
    missing = con.sql("""
        SELECT COUNT(*) AS null_client_ids
        FROM 'hf://datasets/FlyRank/internship-warehouse/**/*.parquet'
        WHERE month = '2026-03' AND client_hash_id IS NULL
    """).df()
    print("\n=== Missing client_hash_id ===")
    print(missing)
except Exception as e:
    print("Adjust once schema confirmed:", e)

=== Files in warehouse ===
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_con

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
This slice can't tell us about clients without GSC/GA4 access
(access_profile shows some clients have no_search_or_analytics_access),
so those clients are structurally invisible to this lane. It also can't
distinguish seasonal effects from genuine decline within a single
90-day-style window, and the final month is a sealed test window I'm
not allowed to use for label logic.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.